
---
SECTION 1 – Python & Async Deep Dive (10 minutes)
---
---


Q1 (MCQ)

Which statements about asyncio are TRUE? (Select ALL)

A. asyncio.gather() runs coroutines sequentially

B. CPU-bound work should be moved off the event loop

C. await always creates a new thread

D. Blocking I/O inside async code reduces concurrency

**Answer:**

B, D

Explanation:
- asyncio.gather() takes a set of coroutines or tasks, creates tasks out of the coroutines thus registering them to the
active event loop and awaits them. Therefore it runs them all asynchronously and not sequentially since await stops operation of a task until
resource is ready and lets the event loop continue with other operations.

- CPU bound work is always blocking the event loop in python because of GIL.

- await does not create a thread (a concurrent async unit created and managed by the OS inside a process and passed to python),
it creates a Task (created and managed by python) that registers the coroutine to the event loop and executes it.

- A blocking operation won't yield control back to the event loop even if awaited so the entire event loop would be stuck until it's completed. 




---

Q2 (Code Reasoning)

What is the problem with this code?
```python
@app.get("/items")
async def get_items():
    items = requests.get("https://example.com/data").json()
    return items
```

Explain what’s wrong and how to fix it.

**Answer:**

The path function calls a synchronous http library so it blocks the entire event loop making the entire worker synchronous.
This negates the benefit of working with FastAPI that allows workers to run their own event loops and handle io bound tasks concurrently.
To fix this, we need to use an async http library such as httpx to send a get request in a non blocking way such as:

```python
with httpx.AsyncClient() as client:
    items: list[Item] = await client.get(...)
```



---

Q3 (Short Answer)

When would you intentionally choose sync endpoints (def) in FastAPI instead of async def?

**Answer:**

In cases where we run blocking operations such as CPU/OS bound operations without any need to await async operations
it would be better to not use an async function to not deal with the overhead of wrapping a coroutine with a task and registering it to the event loop that would get blocked anyway because of the synchronous operation.




---
SECTION 2 – FastAPI Architecture & Dependencies (15 minutes)
---
---



Q4 (MCQ)

Which dependency definition guarantees cleanup after request completion?
```python
def get_db():
    db = Session()
    try:
        yield db
    finally:
        db.close()
```

A. Middleware

B. Startup event

C. Dependency with yield

D. Background task

**Answer:**

C. The dependency itself needs to have the cleanup logic in it to ensure cleanup every time (that's why middleware won't ensure it).
    In order to ensure that there would be cleanup after each response is sent, we need a finally block after yielding the resource
    (in case of an error fast api would catch it implicitly and send it in a response).


---

Q5 (Design)

You need to share:

DB session

authenticated user

request ID


across many endpoints.

How would you structure this using FastAPI dependencies?

**Answer:**

I would create a dependency_layer.py module that wraps each resource's provider/factory function (fetch_session, user_auth, ...)
with a Depends class. I would then import the relevant Depends(factory_func) in the path functions that need it.



---

Q6 (Code Completion)

Complete this dependency so it:

extracts JWT from Authorization header

raises 401 if missing

```python
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

def get_current_user(token: str = Depends(____)):
    if not token:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED
        )
    return token
```

**Answer:**


```python
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

def get_current_user(token: str = Depends(oauth2_scheme)):
    if not token:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED
        )
    return token
```



---

Q7 (Conceptual)

What is the difference between:

middleware

dependencies


and when should each be used?

**Answer:**

Middleware - Runs before the path function is called and after the path function finishes. The operation of the middleware is the same for 
every endpoint independent of the operation of the path functions themselves.

Dependencies - Injected to path functions according to the definition of each path function. A dependency is a specific resource or operation 
that are created or fetched when the endpoint is accessed so they are endpoint specific and run before and after the path function is called.


---

SECTION 3 – Databases, Performance & Scaling (10 minutes)



Q8 (MCQ)

Which choices help prevent DB exhaustion in high-load systems?

A. One global DB connection

B. Connection pooling

C. Async DB drivers

D. Per-request DB engine creation


**Answer:**
B, C




---

Q9 (Scenario)

You notice increasing latency under load even though CPU usage is low.

Name three possible backend causes and how you’d investigate them.

**Answer:**

1. **Blocking operation:** Running an CPU bound process / syncronous io bound operation that blocks the event loop of one or more workers.
    </br>
    I would use a profiler that could point to the bottleneck and go over the flow step by step with a debugger.

2. **DB connection limit:** Assuming we are working with an async db conenctions (otherwise it's the previous reason) creation of too many connections and / or not managing an async connection pool.
    </br>
    I would investigate it by viewing the active connections on the database / resource side and investigating the code making sure we are
    working with a connectin pool of async connections.

3. **Rate limit reached / bad cleanup or termination:** We might have reached the request rate limit for an external API or we have unterminated connection or resources that inflate the memory signature of our app even though the CPU usage is low.
</br>
I would investigate this by checking the web responses with a debugger, examining the errors I get in addition to monitoring my request rate to learn the limit pattern of the API I'm connecting to. I would also use a profiler to examine the memory signature of my app.


---

Q10 (Design)

Explain how you would implement:

idempotent POST requests

in a distributed API system

**Answer:** If I have control over the API I'm connecting to I would make sure it adds a uuid as a idempotency key in the post requests it sends.
    </br>
    I would then extract this key from an idempotency key header in the post request and store it together with the data from the payload.
    </br>
    When storing the data I would check if the idempotency key exist in my db and only store the data from the post request if it doesn't.



---

SECTION 4 – Security, Cloud & CI/CD (15 minutes)

Q11 (MCQ)

Which JWT claim is most important for token expiration?

A. iss

B. aud

C. exp

D. sub

**Answer:** C, token expiration.


---

Q12 (Scenario)

A FastAPI service calls OpenAI APIs. What security and reliability safeguards should be in place?

List at least 4.

**Answer:**

1. Protocol security - we must use HTTPS to prevent man in the middle attacks.

2. Using a message queue to query the API - To prevent the same requst from being sent multiple times, we should use a streaming service like 
    </br>
    Kafka to manage the reliablity and load of the requests.

3. Using secure authentication to the AI provider - We need to make sure to use a token based / api key based communication to prevent forgery
    </br>
    and XSS from using our credentials querying the API (using our money) and scripts impersonating the AI provider collecting our prompts with
    </br>
    proprietary data.

4. Strict validation of the response structure and data format - The responses from ai models can vary and data in the wrong format or size
    </br>
    could break our backend so we should use middleware that checks the format of the response (no binary files or executables etc)
    </br>
    and pydantic models for the content of the response from the AI to verify that the content is digestable by our system.



---

Q13 (Docker)

Why is this pattern preferred?
```dockerfile
RUN pip install --no-cache-dir -r requirements.txt
```

**Answer:**
the cache dir in pip is useful for storing artifacts needed for the installation such a wheel, tarballs etc. in case the installation fails ane we need to re install or we want to install packages in more than one virtual environment.
</br>
I the case of building a docker image it's useless since the **stages (FROM ...)** of the image are separate so the cache dir is not reused even if we wanted
</br>
to install packages multiple times during the build of the image. Having redundant data in pip's cache dir increases the size of the image
</br>
and the attack surface on it since there are more sources for malicious files saved in the image.




---

Q14 (CI/CD)

What checks would you block merges on for a senior backend role?

List at least 4.

**Answer:**

1. Functionality - unit, regression and integration testing.
</br>

2. Code Quality - Static linting, important for maintainability and performance.
</br>

3. Performance - Profiling (memory, CPU).
</br>

4. Security - Static and dynamic security tests, vulnerability (dependencies, protocol).
</br>


---

SECTION 5 – AI/ML Integration & System Design (10 minutes)



Q15 (Design)

Design a FastAPI endpoint that:

accepts user input

calls an AI model

returns results

does not block the event loop

handles failures gracefully


Describe components, not code.

**Answer:**
I would design the system in a clean architecture pattern.

1. Presentation layer -
    </br>
    Entry point (main.py)
    </br>
    dependency injection (dependency_layer.py, uses Depends and/or dependency_injector) 
    </br>
    server_and_middlewares.py - Registering middlewares and setup of the backend server.

2. Application layer - 
    </br>
    sign_in.py (uses FastAPI, Celery) - Endpoint for user sign in with background task wrapper for the authentication domain service.
    </br>
    handle_prompt.py (uses FastAPI) - Endpoint that recieves a post request, gets the response body as a pydantic model, creates a request,
    </br>
    and sends it asynchronously (using httpx) to the model provider via await IAIProvider.send_prompt(context).
    </br>
    It then takes the response, turns it into a pydantic model for validation, creates a request from it and passes it to the front end.

3. Domain layer -
    </br>
    models.py - Pydantic models for strict typing and data validation of request body, response body, users and DTOs.
    </br>
    authorisation.py - Logic for JWT signature validation (pyJWT).
    </br>
    services.py - Processing the request and response content, user authentication service.

4. Infrastructure layer -
    </br>
    storage.py (PostgresRepository, RedisRepository - protocol)
    </br>
    data_brokers.py (RabbitMQ)
    </br>
    task_management.py (Celery) - Config for Celery.
    </br>
    ai.py (LiteLLM) - Connects to the model provider, generates prompt, returns response in raw form.
    </br>
    interfaces.py - IStorageProvider, IAIProvider


---

Q16 (Architecture)

Where should prompt logic live in a clean backend architecture?

A. API route

B. Database layer

C. Service/domain layer

D. Frontend

Explain briefly.


**Answer:** C.
</br>
I would create a pydantic model for the structure of the prompt (context, metadada, format) in domain/models.py for safety and token consumption optimisation.
</br>
The actual generation of the context of the prompt should be part of a domain service that runs business logic. For instance domain.services.ai.LLM.generate_prompt()

Q17 (Trade-offs)

When would you prefer:

synchronous inference

vs background jobs (Celery / queues)?

**Answer:**
I would prefer **synchronous inference** in those cases:
- Simple tasks - For cases where I don't need to run multiple concurrent background tasks and the tasks are reliable and not very error prone.
- Low latency - When the priority is to get immediate response.

I would prefer **background jobs** when:
- Error prone tasks - I would prefer using Celery + RabbitMQ to manage retries, load balancing, task management and eror handling.
- Multiple concurrent CPU bound tasks - Celery handles multiprocessing in a reliable way so I would prefer using it with RabbitMQ as broker.
- Multiple concurrent I/O bound background tasks - FastAPI's BackgroundTask class would be my choice since it registers the tasks to the event loop.